# Data Cleaning — TED eForms raw data (19 file CSV)

Notebook này làm sạch **từng file CSV riêng biệt** trong `database/raw_data/` (dữ liệu tender xuất từ
oeffentlichevergabe.de theo chuẩn eForms/TED). Mỗi file là **1 DataFrame độc lập** (`df_<ten_file>`) —
**không merge/join** ở đây, việc ghép dữ liệu giữa các file sẽ làm ở bước ETL riêng sau.

## Nhóm file xử lý

- **19 file eForms** (Section 1–19): `notice`, `procedure`, `purpose`, `classification`,
  `placeOfPerformance`, `duration`, `submissionTerms`, `organisation`, `lot`, `tender`, `contract`,
  `procedureLotResult`, `receivedSubmissions`, `noticeResult`, `changes`, `secondStage`,
  `cvdInformation`, `additionalInformation`, `strategicProcurement` — làm sạch đầy đủ: xoá trùng lặp,
  chuẩn hoá kiểu số/ngày/boolean, xử lý thiếu dữ liệu ở cột khoá, strip text.

`TED_17-09-2026.csv` đã bị xoá khỏi `raw_data/` và không còn được xử lý trong notebook này.

## ⚠️ Lưu ý chung quan trọng — tránh xoá nhầm dữ liệu hợp lệ

Ở nhiều file (`purpose`, `classification`, `placeOfPerformance`), cột `lotIdentifier` **rỗng có chủ
đích**: mỗi gói thầu có 1 dòng "cấp gói thầu" (`lotIdentifier` rỗng) và thêm 1 dòng nữa cho mỗi lot con
(vd `LOT-0000`). Vì vậy **không được** coi `lotIdentifier` rỗng ở 3 file này là "thiếu dữ liệu" rồi xoá
dòng — chỉ `noticeIdentifier`/`noticeVersion` (và `lotIdentifier` ở các file mà nó luôn bắt buộc như
`duration`, `submissionTerms`, `lot`) mới được xử lý theo logic "thiếu khoá bắt buộc -> xoá dòng".

Ngoài ra, cột `noticeVersion` ở **mọi file** đều toàn chữ số (vd `'01'`, `'1'`) nên nếu đọc CSV không ép kiểu, pandas sẽ tự suy luận thành số nguyên và **làm mất số 0 ở đầu** (`'01'` -> `1`), phá hỏng giá trị khoá gốc. Vì vậy mọi lệnh `pd.read_csv` trong notebook này đều dùng `dtype=KEY_DTYPE` (định nghĩa ở Section 0) để ép `noticeIdentifier`/`noticeVersion`/`lotIdentifier` về kiểu chuỗi ngay khi đọc.

In [10]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
np.seterr(divide='ignore', invalid='ignore', over='ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

RAW_DIR = 'database/raw_data/'
CLEAN_DIR = 'database/raw_data/cleaned/'
import os
os.makedirs(CLEAN_DIR, exist_ok=True)

# Cac cot khoa (id/version/lot) phai doc voi dtype=str: neu de pandas tu suy luan kieu,
# gia tri dang chuoi so nhu '01' se bi hieu nham thanh so 1 (mat so 0 dau), lam sai
# du lieu khoa dung de join o buoc ETL sau. Dict nay dung chung cho moi file: pandas se
# tu bo qua cac key khong ton tai trong file dang doc, nen dung an toan cho ca 20 file.
KEY_DTYPE = {'noticeIdentifier': str, 'noticeVersion': str, 'lotIdentifier': str}

# Danh sach tong hop ket qua clean cua TAT CA file (dung cho bang tom tat o section cuoi)
summary_records = []


## 0.1 Hàm dùng chung

Các hàm tiện ích dùng lại cho nhiều DataFrame khác nhau (mỗi DataFrame vẫn được xử lý độc lập, các hàm
này chỉ giúp tránh lặp code cho các thao tác chuẩn hoá kiểu dữ liệu giống nhau).

In [11]:
def strip_text_columns(df, columns):
    """Loại khoảng trắng thừa ở đầu/cuối cho các cột dạng text; chuỗi rỗng sau khi strip -> NaN."""
    for col in columns:
        df[col] = df[col].astype('string').str.strip()
        df[col] = df[col].replace('', pd.NA)
    return df


def to_bool(series):
    """Chuyển cột dạng chuỗi 'true'/'false' về kiểu boolean có thể NaN (nullable boolean)."""
    mapping = {'true': True, 'false': False}
    return series.astype('string').str.strip().str.lower().map(mapping).astype('boolean')


def to_datetime_col(series):
    """Chuyển cột ngày giờ dạng ISO 8601 (có timezone) về kiểu datetime; giá trị lỗi -> NaT."""
    return pd.to_datetime(series, errors='coerce', utc=True)


def to_numeric_col(series):
    """Chuyển cột dạng số (có thể lẫn khoảng trắng thừa) về kiểu numeric; giá trị lỗi -> NaN."""
    return pd.to_numeric(series.astype('string').str.strip(), errors='coerce')


def drop_missing_keys(df, key_columns, label):
    """Xoá các dòng thiếu giá trị ở cột khoá bắt buộc (vd noticeIdentifier), in số dòng đã loại."""
    n_before = len(df)
    df = df.dropna(subset=key_columns)
    n_after = len(df)
    if n_before != n_after:
        print(f"[{label}] Đã xoá {n_before - n_after} dòng thiếu khoá bắt buộc {key_columns}")
    return df


## 1. `notice.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `procedureIdentifier`, `procedureLegalBasis`, `formType`,
`noticeType`, `publicationDate`. Không có sai khác so với giả định ban đầu.

- `procedureIdentifier` thiếu ở ~38% dòng — đây là FK tuỳ chọn (không phải notice nào cũng có thủ tục
  liên kết sẵn), giữ nguyên NaN, không xoá dòng.
- `publicationDate` là chuỗi ISO 8601 kèm timezone -> chuẩn hoá về `datetime`.

In [12]:
# --- Load: notice.csv ---
fpath = RAW_DIR + 'notice.csv'
df_notice = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_notice.shape}")

print("Info:")
df_notice.info()
print("\nHead:")
print(df_notice.head())
print("\nMissing values:")
print(df_notice.isnull().sum())
print("\nDescribe:")
print(df_notice.describe(include='all'))

n_before = df_notice.shape[0]

# --- Clean: notice.csv ---
# Xoá dòng trùng lặp hoàn toàn
df_notice = df_notice.drop_duplicates()

# Xoá dòng thiếu khoá bắt buộc (noticeIdentifier/noticeVersion)
df_notice = drop_missing_keys(df_notice, ['noticeIdentifier', 'noticeVersion'], 'notice')

# Chuẩn hoá các cột text: loại khoảng trắng thừa, chuỗi rỗng -> NaN
df_notice = strip_text_columns(
    df_notice, ['noticeIdentifier', 'noticeVersion', 'procedureIdentifier',
                'procedureLegalBasis', 'formType', 'noticeType']
)

# Chuẩn hoá cột ngày giờ (ISO 8601 kèm timezone) về kiểu datetime
df_notice['publicationDate'] = to_datetime_col(df_notice['publicationDate'])

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_notice.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_notice.shape[0]}")

# --- Save ---
df_notice.to_csv(CLEAN_DIR + 'notice_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}notice_clean.csv")

summary_records.append({
    'file': 'notice.csv', 'rows_before': n_before, 'rows_after': df_notice.shape[0],
    'rows_removed': n_before - df_notice.shape[0],
})


FileNotFoundError: [Errno 2] No such file or directory: 'database/raw_data/notice.csv'

## 2. `procedure.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `crossBorderLaw`, `procedureType`, `procedureFeatures`,
`procedureAccelerated`, `lotsMaxAllowed`, `lotsAllRequired`, `lotsMaxAwarded`. Không sai khác so với giả
định ban đầu.

- `procedureAccelerated`, `lotsAllRequired` là chuỗi `'true'`/`'false'` -> chuẩn hoá về boolean.
- `lotsMaxAllowed`, `lotsMaxAwarded` là số nguyên dạng chuỗi (thiếu ~95%, hầu hết thủ tục không chia lot
  hoặc không giới hạn) -> chuẩn hoá về kiểu số nguyên có thể NaN (`Int64`).

In [ ]:
# --- Load: procedure.csv ---
fpath = RAW_DIR + 'procedure.csv'
df_procedure = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_procedure.shape}")

print("Info:")
df_procedure.info()
print("\nHead:")
print(df_procedure.head())
print("\nMissing values:")
print(df_procedure.isnull().sum())
print("\nDescribe:")
print(df_procedure.describe(include='all'))

n_before = df_procedure.shape[0]

# --- Clean: procedure.csv ---
df_procedure = df_procedure.drop_duplicates()
df_procedure = drop_missing_keys(df_procedure, ['noticeIdentifier', 'noticeVersion'], 'procedure')

df_procedure = strip_text_columns(
    df_procedure, ['noticeIdentifier', 'noticeVersion', 'crossBorderLaw', 'procedureType', 'procedureFeatures']
)

# Chuẩn hoá 2 cột boolean dạng chuỗi 'true'/'false'
df_procedure['procedureAccelerated'] = to_bool(df_procedure['procedureAccelerated'])
df_procedure['lotsAllRequired'] = to_bool(df_procedure['lotsAllRequired'])

# Chuẩn hoá 2 cột số nguyên (số lượng lot tối đa) về kiểu Int64 (cho phép NaN)
df_procedure['lotsMaxAllowed'] = to_numeric_col(df_procedure['lotsMaxAllowed']).astype('Int64')
df_procedure['lotsMaxAwarded'] = to_numeric_col(df_procedure['lotsMaxAwarded']).astype('Int64')

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_procedure.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_procedure.shape[0]}")

# --- Save ---
df_procedure.to_csv(CLEAN_DIR + 'procedure_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}procedure_clean.csv")

summary_records.append({
    'file': 'procedure.csv', 'rows_before': n_before, 'rows_after': df_procedure.shape[0],
    'rows_removed': n_before - df_procedure.shape[0],
})


## 3. `purpose.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `lotIdentifier`, `internalIdentifier`, `mainNature`,
`additionalNature`, `title`, `estimatedValue`, `estimatedValueCurrency`, `description`. Không sai khác so
với giả định ban đầu.

- `lotIdentifier` rỗng ở ~43% dòng **có chủ đích** (dòng cấp gói thầu tổng, không phải lot con) -> **không**
  đưa vào danh sách khoá bắt buộc để xoá dòng, chỉ `noticeIdentifier`/`noticeVersion` mới bắt buộc.
- `estimatedValue` thiếu ~93% (đa số gói thầu không công khai giá trị dự kiến ở cấp này) -> chuẩn hoá kiểu
  số nhưng **giữ nguyên NaN**  , không tự điền giá trị giả.

In [ ]:
# --- Load: purpose.csv ---
fpath = RAW_DIR + 'purpose.csv'
df_purpose = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_purpose.shape}")

print("Info:")
df_purpose.info()
print("\nHead:")
print(df_purpose.head())
print("\nMissing values:")
print(df_purpose.isnull().sum())
print("\nDescribe:")
print(df_purpose.describe(include='all'))

n_before = df_purpose.shape[0]

# --- Clean: purpose.csv ---
df_purpose = df_purpose.drop_duplicates()

# Chỉ noticeIdentifier/noticeVersion là khoá bắt buộc; lotIdentifier rỗng là hợp lệ
# (đại diện dòng cấp gói thầu tổng) nên KHÔNG đưa vào danh sách khoá bắt buộc ở đây
df_purpose = drop_missing_keys(df_purpose, ['noticeIdentifier', 'noticeVersion'], 'purpose')

df_purpose = strip_text_columns(
    df_purpose, ['noticeIdentifier', 'noticeVersion', 'lotIdentifier', 'internalIdentifier',
                 'mainNature', 'additionalNature', 'title', 'description', 'estimatedValueCurrency']
)

# Chuẩn hoá cột giá trị tiền tệ về kiểu số (giữ NaN nếu thiếu, không tự điền giá trị giả)
df_purpose['estimatedValue'] = to_numeric_col(df_purpose['estimatedValue'])

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_purpose.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_purpose.shape[0]}")

# --- Save ---
df_purpose.to_csv(CLEAN_DIR + 'purpose_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}purpose_clean.csv")

summary_records.append({
    'file': 'purpose.csv', 'rows_before': n_before, 'rows_after': df_purpose.shape[0],
    'rows_removed': n_before - df_purpose.shape[0],
})


## 4. `classification.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `lotIdentifier`, `classificationType`,
`mainClassificationCode`, `additionalClassificationCodes`, `options`. Không sai khác so với giả định ban
đầu.

- `lotIdentifier` rỗng ~43% có chủ đích (dòng cấp gói thầu) — không coi là thiếu dữ liệu.
- `mainClassificationCode` (mã CPV) **giữ nguyên dạng text**, không ép kiểu số — mã CPV có thể có định
  dạng không đồng nhất (mã 8 số chuẩn, mã kèm số kiểm tra, nhiều mã gộp 1 ô...); việc trích xuất/chuẩn hoá
  sâu mã CPV sẽ làm ở bước ETL khi ghép dữ liệu, ở đây chỉ strip khoảng trắng thừa.
- `options` gần như rỗng hoàn toàn (100%) — vẫn giữ cột, chỉ strip như các cột text khác.

In [ ]:
# --- Load: classification.csv ---
fpath = RAW_DIR + 'classification.csv'
df_classification = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_classification.shape}")

print("Info:")
df_classification.info()
print("\nHead:")
print(df_classification.head())
print("\nMissing values:")
print(df_classification.isnull().sum())
print("\nDescribe:")
print(df_classification.describe(include='all'))

n_before = df_classification.shape[0]

# --- Clean: classification.csv ---
df_classification = df_classification.drop_duplicates()
df_classification = drop_missing_keys(df_classification, ['noticeIdentifier', 'noticeVersion'], 'classification')

df_classification = strip_text_columns(
    df_classification, ['noticeIdentifier', 'noticeVersion', 'lotIdentifier', 'classificationType',
                         'mainClassificationCode', 'additionalClassificationCodes', 'options']
)

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_classification.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_classification.shape[0]}")

# --- Save ---
df_classification.to_csv(CLEAN_DIR + 'classification_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}classification_clean.csv")

summary_records.append({
    'file': 'classification.csv', 'rows_before': n_before, 'rows_after': df_classification.shape[0],
    'rows_removed': n_before - df_classification.shape[0],
})


## 5. `placeOfPerformance.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `lotIdentifier`, `placePerformanceCity`,
`placePerformancePostCode`, `placePerformanceCountrySubdivision`, `placePerformanceCountryCode`. Không
sai khác so với giả định ban đầu.

- `lotIdentifier` rỗng ~43% có chủ đích (dòng cấp gói thầu) — không coi là thiếu dữ liệu.
- Đây là file **duy nhất trong nhóm 9 file có dòng trùng lặp hoàn toàn ở dữ liệu gốc (98 dòng)** —
  `drop_duplicates()` xử lý trực tiếp trường hợp này.
- `placePerformanceCity` đôi khi bị dính mã bưu điện vào đầu tên (vd `"01067 Dresden"`) hoặc chứa giá trị
  rác (`"."`) -> tách mã bưu điện, viết hoa chữ đầu nhất quán.

In [ ]:
# --- Load: placeOfPerformance.csv ---
fpath = RAW_DIR + 'placeOfPerformance.csv'
df_placeOfPerformance = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_placeOfPerformance.shape}")

print("Info:")
df_placeOfPerformance.info()
print("\nHead:")
print(df_placeOfPerformance.head())
print("\nMissing values:")
print(df_placeOfPerformance.isnull().sum())
print("\nDescribe:")
print(df_placeOfPerformance.describe(include='all'))

n_before = df_placeOfPerformance.shape[0]

# --- Clean: placeOfPerformance.csv ---
# File này có dòng trùng lặp hoàn toàn ở dữ liệu gốc -> drop_duplicates xử lý trực tiếp
df_placeOfPerformance = df_placeOfPerformance.drop_duplicates()

df_placeOfPerformance = drop_missing_keys(
    df_placeOfPerformance, ['noticeIdentifier', 'noticeVersion'], 'placeOfPerformance'
)

df_placeOfPerformance = strip_text_columns(
    df_placeOfPerformance, ['noticeIdentifier', 'noticeVersion', 'lotIdentifier', 'placePerformancePostCode']
)

import re

def clean_city(value):
    """Chuẩn hoá tên thành phố: bỏ mã bưu điện dính ở đầu, khoảng trắng thừa, viết hoa chữ đầu."""
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    # Một số dòng bị dính mã bưu điện vào trước tên thành phố, vd '01067 Dresden' -> 'Dresden'
    text = re.sub(r'^\d{4,5}\s+', '', text).strip()
    text = re.sub(r'\s+', ' ', text)
    # Giá trị rác/placeholder không mang ý nghĩa -> coi như thiếu dữ liệu
    if text in ('', '.', '-', 'N/A', 'n/a'):
        return pd.NA
    return text.title()

df_placeOfPerformance['placePerformanceCity'] = df_placeOfPerformance['placePerformanceCity'].apply(clean_city)

# Mã vùng (NUTS) và mã quốc gia quy ước viết hoa toàn bộ, vd 'DE300', 'DEU'
df_placeOfPerformance['placePerformanceCountrySubdivision'] = (
    df_placeOfPerformance['placePerformanceCountrySubdivision'].astype('string').str.strip().str.upper()
)
df_placeOfPerformance['placePerformanceCountryCode'] = (
    df_placeOfPerformance['placePerformanceCountryCode'].astype('string').str.strip().str.upper()
)

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_placeOfPerformance.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_placeOfPerformance.shape[0]}")

# --- Save ---
df_placeOfPerformance.to_csv(CLEAN_DIR + 'placeOfPerformance_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}placeOfPerformance_clean.csv")

summary_records.append({
    'file': 'placeOfPerformance.csv', 'rows_before': n_before, 'rows_after': df_placeOfPerformance.shape[0],
    'rows_removed': n_before - df_placeOfPerformance.shape[0],
})


## 6. `duration.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `lotIdentifier`, `durationStartDate`, `durationPeriod`,
`durationPeriodUnit`, `durationEndDate`, `durationOther`, `renewalMaximum`. Không sai khác so với giả
định ban đầu.

- Khác với `purpose`/`classification`/`placeOfPerformance`, ở file này **`lotIdentifier` luôn có giá trị**
  (0% thiếu trong dữ liệu khảo sát) — mỗi dòng gắn với đúng 1 lot cụ thể, nên được coi là khoá bắt buộc.
- `durationStartDate`/`durationEndDate` là chuỗi ISO 8601 kèm timezone -> chuẩn hoá `datetime`.
- `durationOther` chứa giá trị text có ý nghĩa thật (vd `'UNKNOWN'`, `'UNLIMITED'`), không phải placeholder
  rác -> chỉ strip, không suy diễn thành thiếu dữ liệu.

In [ ]:
# --- Load: duration.csv ---
fpath = RAW_DIR + 'duration.csv'
df_duration = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_duration.shape}")

print("Info:")
df_duration.info()
print("\nHead:")
print(df_duration.head())
print("\nMissing values:")
print(df_duration.isnull().sum())
print("\nDescribe:")
print(df_duration.describe(include='all'))

n_before = df_duration.shape[0]

# --- Clean: duration.csv ---
df_duration = df_duration.drop_duplicates()

# lotIdentifier ở file này luôn bắt buộc (mỗi dòng gắn với đúng 1 lot cụ thể)
df_duration = drop_missing_keys(
    df_duration, ['noticeIdentifier', 'noticeVersion', 'lotIdentifier'], 'duration'
)

df_duration = strip_text_columns(
    df_duration, ['noticeIdentifier', 'noticeVersion', 'lotIdentifier', 'durationPeriodUnit', 'durationOther']
)

# Đơn vị thời gian quy ước viết hoa, vd 'day' -> 'DAY'
df_duration['durationPeriodUnit'] = df_duration['durationPeriodUnit'].str.upper()

# Chuẩn hoá 2 cột ngày giờ về kiểu datetime
df_duration['durationStartDate'] = to_datetime_col(df_duration['durationStartDate'])
df_duration['durationEndDate'] = to_datetime_col(df_duration['durationEndDate'])

# Chuẩn hoá cột số (thời lượng) và số nguyên (số lần gia hạn tối đa)
df_duration['durationPeriod'] = to_numeric_col(df_duration['durationPeriod'])
df_duration['renewalMaximum'] = to_numeric_col(df_duration['renewalMaximum']).astype('Int64')

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_duration.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_duration.shape[0]}")

# --- Save ---
df_duration.to_csv(CLEAN_DIR + 'duration_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}duration_clean.csv")

summary_records.append({
    'file': 'duration.csv', 'rows_before': n_before, 'rows_after': df_duration.shape[0],
    'rows_removed': n_before - df_duration.shape[0],
})


## 7. `submissionTerms.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `lotIdentifier`, `tenderValidityDeadline`,
`tenderValidityDeadlineUnit`, `guaranteeRequired`, `publicOpeningDate`. Không sai khác so với giả định
ban đầu.

- `lotIdentifier` luôn có giá trị ở file này (giống `duration.csv`) -> coi là khoá bắt buộc.
- `guaranteeRequired` là chuỗi `'true'`/`'false'` -> chuẩn hoá boolean.
- `publicOpeningDate` là chuỗi ISO 8601 kèm timezone -> chuẩn hoá `datetime`.

In [ ]:
# --- Load: submissionTerms.csv ---
fpath = RAW_DIR + 'submissionTerms.csv'
df_submissionTerms = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_submissionTerms.shape}")

print("Info:")
df_submissionTerms.info()
print("\nHead:")
print(df_submissionTerms.head())
print("\nMissing values:")
print(df_submissionTerms.isnull().sum())
print("\nDescribe:")
print(df_submissionTerms.describe(include='all'))

n_before = df_submissionTerms.shape[0]

# --- Clean: submissionTerms.csv ---
df_submissionTerms = df_submissionTerms.drop_duplicates()

df_submissionTerms = drop_missing_keys(
    df_submissionTerms, ['noticeIdentifier', 'noticeVersion', 'lotIdentifier'], 'submissionTerms'
)

df_submissionTerms = strip_text_columns(
    df_submissionTerms, ['noticeIdentifier', 'noticeVersion', 'lotIdentifier', 'tenderValidityDeadlineUnit']
)
df_submissionTerms['tenderValidityDeadlineUnit'] = df_submissionTerms['tenderValidityDeadlineUnit'].str.upper()

df_submissionTerms['tenderValidityDeadline'] = to_numeric_col(df_submissionTerms['tenderValidityDeadline'])
df_submissionTerms['guaranteeRequired'] = to_bool(df_submissionTerms['guaranteeRequired'])
df_submissionTerms['publicOpeningDate'] = to_datetime_col(df_submissionTerms['publicOpeningDate'])

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_submissionTerms.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_submissionTerms.shape[0]}")

# --- Save ---
df_submissionTerms.to_csv(CLEAN_DIR + 'submissionTerms_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}submissionTerms_clean.csv")

summary_records.append({
    'file': 'submissionTerms.csv', 'rows_before': n_before, 'rows_after': df_submissionTerms.shape[0],
    'rows_removed': n_before - df_submissionTerms.shape[0],
})


## 8. `organisation.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `organisationName`, `organisationIdentifier`,
`organisationCity`, `organisationPostCode`, `organisationCountrySubdivision`, `organisationCountryCode`,
`organisationInternetAddress`, `organisationNaturalPerson`, `organisationRole`, `buyerProfileURL`,
`buyerLegalType`, `buyerContractingEntity`, `winnerSize`, `winnerOwnerNationality`, `winnerListed`.

- Khác với `duration`/`submissionTerms`, file này **không có cột `lotIdentifier`** — quan hệ
  buyer/reviewer/tenderer/winner được gắn ở cấp notice, không phải cấp lot.
- `organisationRole` có 8 giá trị thật: `buyer`, `reviewer`, `tenderer`, `winner`, `mediator`, `subcont`,
  `serv-prov`, `ted-esen`.
- `organisationNaturalPerson`, `buyerContractingEntity`, `winnerListed` là các cột boolean dạng chuỗi
  `'true'`/`'false'` -> chuẩn hoá boolean.
- `organisationCountryCode`, `organisationCountrySubdivision`, `winnerOwnerNationality` là mã quy ước
  viết hoa toàn bộ (vd `'DEU'`) -> chuẩn hoá uppercase.

In [ ]:
# --- Load: organisation.csv ---
fpath = RAW_DIR + 'organisation.csv'
df_organisation = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_organisation.shape}")

print("Info:")
df_organisation.info()
print("\nHead:")
print(df_organisation.head())
print("\nMissing values:")
print(df_organisation.isnull().sum())
print("\nDescribe:")
print(df_organisation.describe(include='all'))

n_before = df_organisation.shape[0]

# --- Clean: organisation.csv ---
df_organisation = df_organisation.drop_duplicates()
df_organisation = drop_missing_keys(df_organisation, ['noticeIdentifier', 'noticeVersion'], 'organisation')

df_organisation = strip_text_columns(
    df_organisation,
    ['noticeIdentifier', 'noticeVersion', 'organisationName', 'organisationIdentifier',
     'organisationCity', 'organisationPostCode', 'organisationCountrySubdivision',
     'organisationCountryCode', 'organisationInternetAddress', 'organisationRole',
     'buyerProfileURL', 'buyerLegalType', 'winnerSize', 'winnerOwnerNationality']
)

# Các cột mã quy ước viết hoa toàn bộ
for col in ['organisationCountrySubdivision', 'organisationCountryCode', 'winnerOwnerNationality']:
    df_organisation[col] = df_organisation[col].str.upper()

# winnerSize là nhãn phân loại (micro/small/medium/large) -> chuẩn hoá viết thường cho nhất quán
df_organisation['winnerSize'] = df_organisation['winnerSize'].str.lower()

# Chuẩn hoá 3 cột boolean dạng chuỗi 'true'/'false'
df_organisation['organisationNaturalPerson'] = to_bool(df_organisation['organisationNaturalPerson'])
df_organisation['buyerContractingEntity'] = to_bool(df_organisation['buyerContractingEntity'])
df_organisation['winnerListed'] = to_bool(df_organisation['winnerListed'])

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_organisation.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_organisation.shape[0]}")

# --- Save ---
df_organisation.to_csv(CLEAN_DIR + 'organisation_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}organisation_clean.csv")

summary_records.append({
    'file': 'organisation.csv', 'rows_before': n_before, 'rows_after': df_organisation.shape[0],
    'rows_removed': n_before - df_organisation.shape[0],
})


## 9. `lot.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `lotIdentifier` — chỉ là bảng đăng ký danh sách lot (registry),
không có cột dữ liệu nghiệp vụ nào khác. Không sai khác so với giả định ban đầu.

- Cả 3 cột đều là khoá bắt buộc (không có cột nào khác để mất mát nếu xoá dòng thiếu khoá).

In [ ]:
# --- Load: lot.csv ---
fpath = RAW_DIR + 'lot.csv'
df_lot = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_lot.shape}")

print("Info:")
df_lot.info()
print("\nHead:")
print(df_lot.head())
print("\nMissing values:")
print(df_lot.isnull().sum())
print("\nDescribe:")
print(df_lot.describe(include='all'))

n_before = df_lot.shape[0]

# --- Clean: lot.csv ---
df_lot = df_lot.drop_duplicates()

# Cả 3 cột đều là khoá bắt buộc ở file này (bảng đăng ký lot, không có cột nghiệp vụ nào khác)
df_lot = drop_missing_keys(df_lot, ['noticeIdentifier', 'noticeVersion', 'lotIdentifier'], 'lot')

df_lot = strip_text_columns(df_lot, ['noticeIdentifier', 'noticeVersion', 'lotIdentifier'])

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_lot.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_lot.shape[0]}")

# --- Save ---
df_lot.to_csv(CLEAN_DIR + 'lot_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}lot_clean.csv")

summary_records.append({
    'file': 'lot.csv', 'rows_before': n_before, 'rows_after': df_lot.shape[0],
    'rows_removed': n_before - df_lot.shape[0],
})


## 10. `tender.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `tenderIdentifier`, `lotIdentifier`, `tenderValue`,
`tenderValueCurrency`, `tenderPaymentValue`, `tenderPaymentValueCurrency`, `tenderPenalties`,
`tenderPenaltiesCurrency`, `tenderRank`, `concessionRevenueUser`, `concessionRevenueUserCurrency`,
`concessionRevenueBuyer`, `concessionRevenueBuyerCurrency`, `countryOrigin`.

- `noticeIdentifier`/`noticeVersion`/`tenderIdentifier`/`lotIdentifier` đều 0% thiếu trong dữ liệu khảo
  sát -> coi cả 4 là khoá bắt buộc (mỗi dòng là 1 tender cụ thể gắn với đúng 1 lot).
- `tenderPaymentValue(Currency)`, `tenderPenalties(Currency)`, `concessionRevenueUser/Buyer(Currency)`,
  `countryOrigin` thiếu 100% trong dữ liệu khảo sát — các trường này chỉ áp dụng cho hợp đồng nhượng
  quyền (concession) hiếm gặp, giữ nguyên cột theo đúng schema eForms, không tự suy diễn hay xoá cột.
- `tenderValue` chuẩn hoá kiểu số; `tenderRank` chuẩn hoá kiểu số nguyên (`Int64`).

In [ ]:
# --- Load: tender.csv ---
fpath = RAW_DIR + 'tender.csv'
df_tender = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_tender.shape}")

print("Info:")
df_tender.info()
print("\nHead:")
print(df_tender.head())
print("\nMissing values:")
print(df_tender.isnull().sum())
print("\nDescribe:")
print(df_tender.describe(include='all'))

n_before = df_tender.shape[0]

# --- Clean: tender.csv ---
df_tender = df_tender.drop_duplicates()

# 4 cột đều là khoá bắt buộc (mỗi dòng = 1 tender cụ thể gắn với đúng 1 lot)
df_tender = drop_missing_keys(
    df_tender, ['noticeIdentifier', 'noticeVersion', 'tenderIdentifier', 'lotIdentifier'], 'tender'
)

df_tender = strip_text_columns(
    df_tender,
    ['noticeIdentifier', 'noticeVersion', 'tenderIdentifier', 'lotIdentifier', 'tenderValueCurrency',
     'tenderPaymentValueCurrency', 'tenderPenaltiesCurrency', 'concessionRevenueUserCurrency',
     'concessionRevenueBuyerCurrency', 'countryOrigin']
)

# Các cột mã tiền tệ/quốc gia quy ước viết hoa toàn bộ
for col in ['tenderValueCurrency', 'tenderPaymentValueCurrency', 'tenderPenaltiesCurrency',
            'concessionRevenueUserCurrency', 'concessionRevenueBuyerCurrency', 'countryOrigin']:
    df_tender[col] = df_tender[col].str.upper()

# Chuẩn hoá các cột giá trị tiền tệ về kiểu số (giữ NaN nếu thiếu)
for col in ['tenderValue', 'tenderPaymentValue', 'tenderPenalties',
            'concessionRevenueUser', 'concessionRevenueBuyer']:
    df_tender[col] = to_numeric_col(df_tender[col])

# tenderRank là số thứ hạng -> chuẩn hoá kiểu số nguyên có thể NaN
df_tender['tenderRank'] = to_numeric_col(df_tender['tenderRank']).astype('Int64')

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_tender.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_tender.shape[0]}")

# --- Save ---
df_tender.to_csv(CLEAN_DIR + 'tender_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}tender_clean.csv")

summary_records.append({
    'file': 'tender.csv', 'rows_before': n_before, 'rows_after': df_tender.shape[0],
    'rows_removed': n_before - df_tender.shape[0],
})


## 11. `contract.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `contractIdentifier`, `winnerDecisionDate`,
`contractConclusionDate`, `contractFrameworkAgreement`.

- `noticeIdentifier`/`noticeVersion`/`contractIdentifier` là khoá bắt buộc (0% thiếu, mỗi dòng là 1 hợp
  đồng cụ thể).
- `winnerDecisionDate` thiếu ~60% (không phải thông báo nào cũng công bố ngày quyết định trúng thầu
  riêng), `contractConclusionDate` thiếu ~3% -> chuẩn hoá cả hai về `datetime`, giữ NaT nếu thiếu.
- `contractFrameworkAgreement` là cột boolean dạng chuỗi `'true'`/`'false'`, thiếu ~99.7% (đa số hợp đồng
  không thuộc khung thoả thuận) -> chuẩn hoá boolean, giữ NaN.

In [ ]:
# --- Load: contract.csv ---
fpath = RAW_DIR + 'contract.csv'
df_contract = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_contract.shape}")

print("Info:")
df_contract.info()
print("\nHead:")
print(df_contract.head())
print("\nMissing values:")
print(df_contract.isnull().sum())
print("\nDescribe:")
print(df_contract.describe(include='all'))

n_before = df_contract.shape[0]

# --- Clean: contract.csv ---
df_contract = df_contract.drop_duplicates()

df_contract = drop_missing_keys(
    df_contract, ['noticeIdentifier', 'noticeVersion', 'contractIdentifier'], 'contract'
)

df_contract = strip_text_columns(df_contract, ['noticeIdentifier', 'noticeVersion', 'contractIdentifier'])

# Chuẩn hoá 2 cột ngày giờ (ISO 8601 kèm timezone) về kiểu datetime
df_contract['winnerDecisionDate'] = to_datetime_col(df_contract['winnerDecisionDate'])
df_contract['contractConclusionDate'] = to_datetime_col(df_contract['contractConclusionDate'])

# Chuẩn hoá cột boolean dạng chuỗi 'true'/'false'
df_contract['contractFrameworkAgreement'] = to_bool(df_contract['contractFrameworkAgreement'])

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_contract.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_contract.shape[0]}")

# --- Save ---
df_contract.to_csv(CLEAN_DIR + 'contract_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}contract_clean.csv")

summary_records.append({
    'file': 'contract.csv', 'rows_before': n_before, 'rows_after': df_contract.shape[0],
    'rows_removed': n_before - df_contract.shape[0],
})


## 12. `procedureLotResult.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `lotIdentifier`, `procedureLotResultNumber`,
`winnerChosen`, `notAwardedReason`, `frameworkMaximumValue`, `frameworkMaximumValueCurrency`,
`frameworkEstimatedValue`, `frameworkEstimatedValueCurrency`, `tenderValueLowest`,
`tenderValueLowestCurrency`, `tenderValueHighest`, `tenderValueHighestCurrency`.

- `noticeIdentifier`/`noticeVersion`/`lotIdentifier`/`procedureLotResultNumber` đều 0% thiếu -> khoá bắt
  buộc (mỗi dòng là kết quả của đúng 1 lot).
- `winnerChosen` (`clos-nw`/`selec-w`) và `notAwardedReason` là nhãn phân loại, chỉ strip text, không ép
  kiểu số.
- Các cột giá trị framework/tender (`frameworkMaximumValue`, `frameworkEstimatedValue`,
  `tenderValueLowest`, `tenderValueHighest`) thiếu 76–99% (hầu hết lot không phải khung thoả thuận hoặc
  không công khai giá thầu thấp/cao nhất) -> chuẩn hoá kiểu số, giữ NaN, không tự điền giá trị giả.

In [ ]:
# --- Load: procedureLotResult.csv ---
fpath = RAW_DIR + 'procedureLotResult.csv'
df_procedureLotResult = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_procedureLotResult.shape}")

print("Info:")
df_procedureLotResult.info()
print("\nHead:")
print(df_procedureLotResult.head())
print("\nMissing values:")
print(df_procedureLotResult.isnull().sum())
print("\nDescribe:")
print(df_procedureLotResult.describe(include='all'))

n_before = df_procedureLotResult.shape[0]

# --- Clean: procedureLotResult.csv ---
df_procedureLotResult = df_procedureLotResult.drop_duplicates()

df_procedureLotResult = drop_missing_keys(
    df_procedureLotResult,
    ['noticeIdentifier', 'noticeVersion', 'lotIdentifier', 'procedureLotResultNumber'],
    'procedureLotResult'
)

df_procedureLotResult = strip_text_columns(
    df_procedureLotResult,
    ['noticeIdentifier', 'noticeVersion', 'lotIdentifier', 'procedureLotResultNumber',
     'winnerChosen', 'notAwardedReason', 'frameworkMaximumValueCurrency',
     'frameworkEstimatedValueCurrency', 'tenderValueLowestCurrency', 'tenderValueHighestCurrency']
)

# Các cột mã tiền tệ quy ước viết hoa toàn bộ
for col in ['frameworkMaximumValueCurrency', 'frameworkEstimatedValueCurrency',
            'tenderValueLowestCurrency', 'tenderValueHighestCurrency']:
    df_procedureLotResult[col] = df_procedureLotResult[col].str.upper()

# Chuẩn hoá các cột giá trị tiền tệ về kiểu số (giữ NaN nếu thiếu)
for col in ['frameworkMaximumValue', 'frameworkEstimatedValue', 'tenderValueLowest', 'tenderValueHighest']:
    df_procedureLotResult[col] = to_numeric_col(df_procedureLotResult[col])

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_procedureLotResult.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_procedureLotResult.shape[0]}")

# --- Save ---
df_procedureLotResult.to_csv(CLEAN_DIR + 'procedureLotResult_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}procedureLotResult_clean.csv")

summary_records.append({
    'file': 'procedureLotResult.csv', 'rows_before': n_before, 'rows_after': df_procedureLotResult.shape[0],
    'rows_removed': n_before - df_procedureLotResult.shape[0],
})


## 13. `receivedSubmissions.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `procedureLotResultNumber`, `receivedSubmissionsCount`,
`receivedSubmissionsType`.

- `noticeIdentifier`/`noticeVersion`/`procedureLotResultNumber` đều 0% thiếu -> khoá bắt buộc.
  `receivedSubmissionsType` thiếu ~55% nhưng đây là dòng breakdown theo loại hồ sơ (`tenders`,
  `t-esubm`, `t-sme`...) nên **không** đưa vào khoá bắt buộc — thiếu nghĩa là dòng tổng, không phải lỗi
  dữ liệu.
- Đây là file **có dòng trùng lặp hoàn toàn ở dữ liệu gốc** (480/1589 dòng, giống `placeOfPerformance.csv`
  ở Section 5) — `drop_duplicates()` xử lý trực tiếp trường hợp này.
- `receivedSubmissionsCount` chuẩn hoá về kiểu số nguyên (`Int64`).

In [ ]:
# --- Load: receivedSubmissions.csv ---
fpath = RAW_DIR + 'receivedSubmissions.csv'
df_receivedSubmissions = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_receivedSubmissions.shape}")

print("Info:")
df_receivedSubmissions.info()
print("\nHead:")
print(df_receivedSubmissions.head())
print("\nMissing values:")
print(df_receivedSubmissions.isnull().sum())
print("\nDescribe:")
print(df_receivedSubmissions.describe(include='all'))

n_before = df_receivedSubmissions.shape[0]

# --- Clean: receivedSubmissions.csv ---
# File này có dòng trùng lặp hoàn toàn ở dữ liệu gốc -> drop_duplicates xử lý trực tiếp
df_receivedSubmissions = df_receivedSubmissions.drop_duplicates()

# receivedSubmissionsType thiếu là dòng tổng (breakdown theo loại) -> không đưa vào khoá bắt buộc
df_receivedSubmissions = drop_missing_keys(
    df_receivedSubmissions, ['noticeIdentifier', 'noticeVersion', 'procedureLotResultNumber'],
    'receivedSubmissions'
)

df_receivedSubmissions = strip_text_columns(
    df_receivedSubmissions,
    ['noticeIdentifier', 'noticeVersion', 'procedureLotResultNumber', 'receivedSubmissionsType']
)

# Chuẩn hoá cột số lượng hồ sơ nhận được về kiểu số nguyên có thể NaN
df_receivedSubmissions['receivedSubmissionsCount'] = to_numeric_col(
    df_receivedSubmissions['receivedSubmissionsCount']
).astype('Int64')

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_receivedSubmissions.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_receivedSubmissions.shape[0]}")

# --- Save ---
df_receivedSubmissions.to_csv(CLEAN_DIR + 'receivedSubmissions_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}receivedSubmissions_clean.csv")

summary_records.append({
    'file': 'receivedSubmissions.csv', 'rows_before': n_before, 'rows_after': df_receivedSubmissions.shape[0],
    'rows_removed': n_before - df_receivedSubmissions.shape[0],
})


## 14. `noticeResult.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `noticeValue`, `noticeValueCurrency`,
`noticeFrameworkValue`, `noticeFrameworkValueCurrency`.

- `noticeIdentifier`/`noticeVersion` là khoá bắt buộc.
- `noticeValue` thiếu ~55%, `noticeFrameworkValue` thiếu ~97% (đa số notice không công khai tổng giá trị
  hoặc không thuộc khung thoả thuận) -> chuẩn hoá kiểu số, giữ NaN.

In [ ]:
# --- Load: noticeResult.csv ---
fpath = RAW_DIR + 'noticeResult.csv'
df_noticeResult = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_noticeResult.shape}")

print("Info:")
df_noticeResult.info()
print("\nHead:")
print(df_noticeResult.head())
print("\nMissing values:")
print(df_noticeResult.isnull().sum())
print("\nDescribe:")
print(df_noticeResult.describe(include='all'))

n_before = df_noticeResult.shape[0]

# --- Clean: noticeResult.csv ---
df_noticeResult = df_noticeResult.drop_duplicates()
df_noticeResult = drop_missing_keys(df_noticeResult, ['noticeIdentifier', 'noticeVersion'], 'noticeResult')

df_noticeResult = strip_text_columns(
    df_noticeResult, ['noticeIdentifier', 'noticeVersion', 'noticeValueCurrency', 'noticeFrameworkValueCurrency']
)

# Các cột mã tiền tệ quy ước viết hoa toàn bộ
df_noticeResult['noticeValueCurrency'] = df_noticeResult['noticeValueCurrency'].str.upper()
df_noticeResult['noticeFrameworkValueCurrency'] = df_noticeResult['noticeFrameworkValueCurrency'].str.upper()

# Chuẩn hoá các cột giá trị tiền tệ về kiểu số (giữ NaN nếu thiếu)
df_noticeResult['noticeValue'] = to_numeric_col(df_noticeResult['noticeValue'])
df_noticeResult['noticeFrameworkValue'] = to_numeric_col(df_noticeResult['noticeFrameworkValue'])

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_noticeResult.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_noticeResult.shape[0]}")

# --- Save ---
df_noticeResult.to_csv(CLEAN_DIR + 'noticeResult_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}noticeResult_clean.csv")

summary_records.append({
    'file': 'noticeResult.csv', 'rows_before': n_before, 'rows_after': df_noticeResult.shape[0],
    'rows_removed': n_before - df_noticeResult.shape[0],
})


## 15. `changes.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `contractIdentifier`, `changeNoticeVersionIdentifier`,
`changeReasonCode`, `changeReasonDescription`.

- `noticeIdentifier`/`noticeVersion`/`changeNoticeVersionIdentifier` đều 0% thiếu -> khoá bắt buộc.
- `contractIdentifier` thiếu ~73% — đây là FK tuỳ chọn (không phải thay đổi nào cũng gắn với 1 hợp đồng
  cụ thể, có thể là thay đổi ở cấp notice/procedure), giữ nguyên NaN, không xoá dòng.
- `changeReasonDescription` là mô tả tự do (free text) -> chỉ strip khoảng trắng thừa.

In [ ]:
# --- Load: changes.csv ---
fpath = RAW_DIR + 'changes.csv'
df_changes = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_changes.shape}")

print("Info:")
df_changes.info()
print("\nHead:")
print(df_changes.head())
print("\nMissing values:")
print(df_changes.isnull().sum())
print("\nDescribe:")
print(df_changes.describe(include='all'))

n_before = df_changes.shape[0]

# --- Clean: changes.csv ---
df_changes = df_changes.drop_duplicates()

# contractIdentifier là FK tuỳ chọn -> KHÔNG đưa vào danh sách khoá bắt buộc
df_changes = drop_missing_keys(
    df_changes, ['noticeIdentifier', 'noticeVersion', 'changeNoticeVersionIdentifier'], 'changes'
)

df_changes = strip_text_columns(
    df_changes,
    ['noticeIdentifier', 'noticeVersion', 'contractIdentifier', 'changeNoticeVersionIdentifier',
     'changeReasonCode', 'changeReasonDescription']
)

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_changes.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_changes.shape[0]}")

# --- Save ---
df_changes.to_csv(CLEAN_DIR + 'changes_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}changes_clean.csv")

summary_records.append({
    'file': 'changes.csv', 'rows_before': n_before, 'rows_after': df_changes.shape[0],
    'rows_removed': n_before - df_changes.shape[0],
})


## 16. `secondStage.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `lotIdentifier`, `minimumCandidates`,
`maximumCandidatesIndicator`, `maximumCandidatesNumber`, `successiveReduction`, `noNegotiationNecessary`.

- `lotIdentifier` luôn có giá trị ở file này (giống `duration.csv`/`submissionTerms.csv`) -> khoá bắt
  buộc cùng `noticeIdentifier`/`noticeVersion`.
- `minimumCandidates`, `maximumCandidatesNumber` thiếu 93–96% (hầu hết thủ tục không giới hạn số ứng viên
  ở giai đoạn 2) -> chuẩn hoá kiểu số nguyên (`Int64`), giữ NaN.
- `maximumCandidatesIndicator`, `successiveReduction`, `noNegotiationNecessary` là 3 cột boolean dạng
  chuỗi `'true'`/`'false'` -> chuẩn hoá boolean.

In [ ]:
# --- Load: secondStage.csv ---
fpath = RAW_DIR + 'secondStage.csv'
df_secondStage = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_secondStage.shape}")

print("Info:")
df_secondStage.info()
print("\nHead:")
print(df_secondStage.head())
print("\nMissing values:")
print(df_secondStage.isnull().sum())
print("\nDescribe:")
print(df_secondStage.describe(include='all'))

n_before = df_secondStage.shape[0]

# --- Clean: secondStage.csv ---
df_secondStage = df_secondStage.drop_duplicates()

# lotIdentifier ở file này luôn bắt buộc (mỗi dòng gắn với đúng 1 lot cụ thể)
df_secondStage = drop_missing_keys(
    df_secondStage, ['noticeIdentifier', 'noticeVersion', 'lotIdentifier'], 'secondStage'
)

df_secondStage = strip_text_columns(df_secondStage, ['noticeIdentifier', 'noticeVersion', 'lotIdentifier'])

# Chuẩn hoá 2 cột số nguyên (số lượng ứng viên tối thiểu/tối đa) về kiểu Int64
df_secondStage['minimumCandidates'] = to_numeric_col(df_secondStage['minimumCandidates']).astype('Int64')
df_secondStage['maximumCandidatesNumber'] = to_numeric_col(df_secondStage['maximumCandidatesNumber']).astype('Int64')

# Chuẩn hoá 3 cột boolean dạng chuỗi 'true'/'false'
df_secondStage['maximumCandidatesIndicator'] = to_bool(df_secondStage['maximumCandidatesIndicator'])
df_secondStage['successiveReduction'] = to_bool(df_secondStage['successiveReduction'])
df_secondStage['noNegotiationNecessary'] = to_bool(df_secondStage['noNegotiationNecessary'])

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_secondStage.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_secondStage.shape[0]}")

# --- Save ---
df_secondStage.to_csv(CLEAN_DIR + 'secondStage_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}secondStage_clean.csv")

summary_records.append({
    'file': 'secondStage.csv', 'rows_before': n_before, 'rows_after': df_secondStage.shape[0],
    'rows_removed': n_before - df_secondStage.shape[0],
})


## 17. `cvdInformation.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `procedureLotResultNumber`, `cvdContractType`,
`vehicleCategory`, `vehicles`, `zeroEmissionVehicles`, `cleanVehicles`.

- File nhỏ nhất (19 dòng, chỉ 6 notice có báo cáo xe sạch theo Clean Vehicles Directive) — tất cả cột
  đều 0% thiếu.
- `noticeIdentifier`/`noticeVersion`/`procedureLotResultNumber` là khoá bắt buộc.
- `cvdContractType`, `vehicleCategory` là nhãn phân loại -> chỉ strip text.
- `vehicles`, `zeroEmissionVehicles`, `cleanVehicles` là số lượng xe -> chuẩn hoá kiểu số nguyên
  (`Int64`).

In [ ]:
# --- Load: cvdInformation.csv ---
fpath = RAW_DIR + 'cvdInformation.csv'
df_cvdInformation = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_cvdInformation.shape}")

print("Info:")
df_cvdInformation.info()
print("\nHead:")
print(df_cvdInformation.head())
print("\nMissing values:")
print(df_cvdInformation.isnull().sum())
print("\nDescribe:")
print(df_cvdInformation.describe(include='all'))

n_before = df_cvdInformation.shape[0]

# --- Clean: cvdInformation.csv ---
df_cvdInformation = df_cvdInformation.drop_duplicates()

df_cvdInformation = drop_missing_keys(
    df_cvdInformation, ['noticeIdentifier', 'noticeVersion', 'procedureLotResultNumber'], 'cvdInformation'
)

df_cvdInformation = strip_text_columns(
    df_cvdInformation,
    ['noticeIdentifier', 'noticeVersion', 'procedureLotResultNumber', 'cvdContractType', 'vehicleCategory']
)

# Chuẩn hoá 3 cột số lượng xe về kiểu số nguyên có thể NaN
df_cvdInformation['vehicles'] = to_numeric_col(df_cvdInformation['vehicles']).astype('Int64')
df_cvdInformation['zeroEmissionVehicles'] = to_numeric_col(df_cvdInformation['zeroEmissionVehicles']).astype('Int64')
df_cvdInformation['cleanVehicles'] = to_numeric_col(df_cvdInformation['cleanVehicles']).astype('Int64')

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_cvdInformation.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_cvdInformation.shape[0]}")

# --- Save ---
df_cvdInformation.to_csv(CLEAN_DIR + 'cvdInformation_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}cvdInformation_clean.csv")

summary_records.append({
    'file': 'cvdInformation.csv', 'rows_before': n_before, 'rows_after': df_cvdInformation.shape[0],
    'rows_removed': n_before - df_cvdInformation.shape[0],
})


## 18. `additionalInformation.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `lotIdentifier`, `suitableForSMEs`.

- `lotIdentifier` luôn có giá trị ở file này (giống `duration.csv`) -> khoá bắt buộc cùng
  `noticeIdentifier`/`noticeVersion`.
- `suitableForSMEs` là cột boolean dạng chuỗi `'true'`/`'false'`, thiếu ~43% -> chuẩn hoá boolean, giữ
  NaN (không suy diễn "thiếu" thành `false`).

In [ ]:
# --- Load: additionalInformation.csv ---
fpath = RAW_DIR + 'additionalInformation.csv'
df_additionalInformation = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_additionalInformation.shape}")

print("Info:")
df_additionalInformation.info()
print("\nHead:")
print(df_additionalInformation.head())
print("\nMissing values:")
print(df_additionalInformation.isnull().sum())
print("\nDescribe:")
print(df_additionalInformation.describe(include='all'))

n_before = df_additionalInformation.shape[0]

# --- Clean: additionalInformation.csv ---
df_additionalInformation = df_additionalInformation.drop_duplicates()

# lotIdentifier ở file này luôn bắt buộc (mỗi dòng gắn với đúng 1 lot cụ thể)
df_additionalInformation = drop_missing_keys(
    df_additionalInformation, ['noticeIdentifier', 'noticeVersion', 'lotIdentifier'], 'additionalInformation'
)

df_additionalInformation = strip_text_columns(
    df_additionalInformation, ['noticeIdentifier', 'noticeVersion', 'lotIdentifier']
)

# Chuẩn hoá cột boolean dạng chuỗi 'true'/'false'
df_additionalInformation['suitableForSMEs'] = to_bool(df_additionalInformation['suitableForSMEs'])

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_additionalInformation.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_additionalInformation.shape[0]}")

# --- Save ---
df_additionalInformation.to_csv(CLEAN_DIR + 'additionalInformation_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}additionalInformation_clean.csv")

summary_records.append({
    'file': 'additionalInformation.csv', 'rows_before': n_before, 'rows_after': df_additionalInformation.shape[0],
    'rows_removed': n_before - df_additionalInformation.shape[0],
})


## 19. `strategicProcurement.csv`

Cột thật: `noticeIdentifier`, `noticeVersion`, `lotIdentifier`, `strategicProcurement`,
`greenProcurementCriteria`, `greenProcurement`, `socialProcurement`, `innovativeProcurement`,
`accessibility`, `cleanVehiclesDirective`, `cvdContractType`.

- `lotIdentifier` luôn có giá trị ở file này -> khoá bắt buộc cùng `noticeIdentifier`/`noticeVersion`.
- `strategicProcurement` đôi khi chứa **nhiều mã gộp trong 1 ô, phân tách bởi dấu phẩy** (vd
  `'env-imp,inn-pur'`) -> giữ nguyên dạng text, chỉ strip khoảng trắng thừa; việc tách thành nhiều
  dòng/cột sẽ làm ở bước ETL nếu cần, không tách ở đây.
- `greenProcurementCriteria`, `greenProcurement`, `socialProcurement`, `innovativeProcurement`,
  `accessibility` là các nhãn phân loại tuỳ chọn, thiếu 93–99.9% (đa số gói thầu không thuộc diện mua sắm
  chiến lược) -> chỉ strip text, giữ NaN.
- `cleanVehiclesDirective` là cột boolean dạng chuỗi `'true'`/`'false'` -> chuẩn hoá boolean.

In [ ]:
# --- Load: strategicProcurement.csv ---
fpath = RAW_DIR + 'strategicProcurement.csv'
df_strategicProcurement = pd.read_csv(fpath, dtype=KEY_DTYPE)
print(f"Loaded {fpath}: {df_strategicProcurement.shape}")

print("Info:")
df_strategicProcurement.info()
print("\nHead:")
print(df_strategicProcurement.head())
print("\nMissing values:")
print(df_strategicProcurement.isnull().sum())
print("\nDescribe:")
print(df_strategicProcurement.describe(include='all'))

n_before = df_strategicProcurement.shape[0]

# --- Clean: strategicProcurement.csv ---
df_strategicProcurement = df_strategicProcurement.drop_duplicates()

# lotIdentifier ở file này luôn bắt buộc (mỗi dòng gắn với đúng 1 lot cụ thể)
df_strategicProcurement = drop_missing_keys(
    df_strategicProcurement, ['noticeIdentifier', 'noticeVersion', 'lotIdentifier'], 'strategicProcurement'
)

# strategicProcurement có thể chứa nhiều mã gộp bởi dấu phẩy -> giữ nguyên, chỉ strip
df_strategicProcurement = strip_text_columns(
    df_strategicProcurement,
    ['noticeIdentifier', 'noticeVersion', 'lotIdentifier', 'strategicProcurement',
     'greenProcurementCriteria', 'greenProcurement', 'socialProcurement', 'innovativeProcurement',
     'accessibility', 'cvdContractType']
)

# Chuẩn hoá cột boolean dạng chuỗi 'true'/'false'
df_strategicProcurement['cleanVehiclesDirective'] = to_bool(df_strategicProcurement['cleanVehiclesDirective'])

# --- Verify sau khi clean ---
print("Missing values sau khi clean:")
print(df_strategicProcurement.isnull().sum())
print(f"Số dòng: trước {n_before} -> sau {df_strategicProcurement.shape[0]}")

# --- Save ---
df_strategicProcurement.to_csv(CLEAN_DIR + 'strategicProcurement_clean.csv', index=False)
print(f"Saved to {CLEAN_DIR}strategicProcurement_clean.csv")

summary_records.append({
    'file': 'strategicProcurement.csv', 'rows_before': n_before, 'rows_after': df_strategicProcurement.shape[0],
    'rows_removed': n_before - df_strategicProcurement.shape[0],
})


## 20. Tổng hợp kết quả làm sạch

Bảng tóm tắt số dòng trước/sau clean và số dòng đã loại bỏ cho **tất cả 19 file eForms** đã xử lý.

In [ ]:
summary_df = pd.DataFrame(summary_records)
summary_df['pct_removed'] = (summary_df['rows_removed'] / summary_df['rows_before'] * 100).round(2)

print("=== Bảng tóm tắt kết quả làm sạch dữ liệu ===")
display(summary_df)

print(f"\nTổng số file đã xử lý: {len(summary_df)}")
print(f"Tổng số dòng trước clean: {summary_df['rows_before'].sum()}")
print(f"Tổng số dòng sau clean: {summary_df['rows_after'].sum()}")
print(f"Tổng số dòng đã loại bỏ: {summary_df['rows_removed'].sum()}")
